# Battery-Charging Tidal Energy Case Study

This case study demonstrates how VITAL can be used to explore a representative non-grid-connected tidal energy system for Sitkana battery charging.

The example walks through:

- rotor performance loading,
- baseline design setup,
- tidal site comparison,
- dynamic rotor simulation,
- physical constraint checking,
- LCOE estimation,
- and design optimization.

The goal is to show how the software helps answer practical questions such as:

- Which site looks most promising?
- Is the design physically feasible?
- What drives the cost of energy?
- Can the design be improved?

The results should be interpreted as screening-level estimates, not final design recommendations. The conclusions depend on the selected rotor data, vessel/platform assumptions, battery-cost assumptions, optimization bounds, and tidal data source.

## 1. Why this tool matters

Tidal energy development requires decisions about both where to deploy a system and how to design it.

A promising battery-charging site should have:

- sufficient tidal flow,
- practical mooring conditions,
- feasible vessel/platform integration,
- and a system configuration that remains physically feasible and economically viable.

VITAL supports early-stage screening by combining:

- site data,
- turbine performance curves,
- vessel/platform properties,
- dynamic simulation,
- constraint checks,
- and levelized cost of energy (LCOE) calculations.

In this Sitkana case study, we focus on a representative non-grid-connected battery-charging system concept.

In [ ]:
import contextlib
import io
import urllib3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from vital.module_tidal import process_tidal_data
from vital.module_rotor import RotorData
from vital.module_rotor_simulation import RotorSimulation
from vital.module_vessel import VesselData
from vital.module_constraint_checker import ConstraintChecker
from vital.module_lcoe import LCOEData, LCOECalculator
from vital.module_lcoe_optimizer import LCOEOptimizer

# Suppress warnings caused by NOAA requests using verify=False internally.
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

plt.style.use("tableau-colorblind10")

## 2. Load rotor performance data

The rotor performance data defines how the turbine behaves as flow conditions change.

The rotor file provides:

- ``TSR``: tip-speed ratio,
- ``Ct``: thrust coefficient, which indicates the hydrodynamic load the rotor places on the system,
- ``Cq``: torque coefficient, which indicates the torque required or produced by the rotor.

VITAL computes the power coefficient internally as:

$$
C_p = TSR \cdot C_q
$$

These curves are used by the simulation to estimate power production and loading.

After loading the rotor data, we add the rotor-specific coefficient functions and operating points to the baseline configuration so ``RotorSimulation`` can use them.

In [ ]:
rotor = RotorData(
    filename="../data/Sitkana_rotor_data_blade_2.txt",
)

print(f"Optimal Cp: {rotor.CpOpt:.4f}")
print(f"Optimal TSR: {rotor.TSROpt:.4f}")
print(f"Estimated TSRmax: {rotor.TSRmax:.4f}")
print(f"Measured TSR range: {rotor.tsr.min():.4f} to {rotor.tsr.max():.4f}")

# Plot over the measured TSR range to avoid emphasizing extrapolated behavior.
tsr_vals = np.linspace(rotor.tsr.min(), rotor.tsr.max(), 200)

fig, ax = plt.subplots(4, 1, figsize=(8, 7), sharex=True)

ax[0].plot(tsr_vals, rotor.get_cp(tsr_vals), label="Cp")
ax[0].axvline(rotor.TSROpt, color="k", linestyle="--", linewidth=1, label="Optimal TSR")
ax[0].set_ylabel("Cp")
ax[0].legend()
ax[0].grid(True, alpha=0.3)

ax[1].plot(tsr_vals, rotor.get_cq(tsr_vals), label="Cq")
ax[1].set_ylabel("Cq")
ax[1].legend()
ax[1].grid(True, alpha=0.3)

ax[2].plot(tsr_vals, rotor.get_ct(tsr_vals), label="Ct")
ax[2].set_ylabel("Ct")
ax[2].legend()
ax[2].grid(True, alpha=0.3)

ax[3].plot(tsr_vals, rotor.get_cpmin(tsr_vals), label="Cpmin")
ax[3].set_ylabel("Cpmin")
ax[3].set_xlabel("TSR")
ax[3].legend()
ax[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

No separate Cpmin file is supplied in this case study, so ``RotorData`` uses the default $C_{p,\min} = -1.0$ for cavitation checks.

## 3. Baseline design assumptions

To keep the comparison clear, we start with a fixed baseline design.

The baseline design includes:

- rotor size,
- rated power,
- hub depth,
- drivetrain settings,
- vessel/platform geometry,
- generator and power-electronics loss assumptions,
- battery capacity,
- and cost assumptions.

Later, we will compare this baseline design across multiple sites and then test whether the design can be improved through optimization.

This Sitkana case study uses generator and power-electronics loss models to represent non-grid-connected battery-charging operation.

In this example, the loss models are defined as Python callables of generator-side speed and torque:

- ``omega_g``: generator-side angular speed in rad/s
- ``tau``: generator torque in N m

The loss models return power losses in watts. These simplified example models help estimate how much power is lost before energy is delivered to the battery or load.

The example loss models are illustrative and should be replaced with validated hardware-specific generator and power-electronics loss models for design studies.

In [ ]:
def generator_loss_model(omega_g, tau):
    """
    Example generator loss model.

    Parameters
    ----------
    omega_g : float or np.ndarray
        Generator-side angular speed (rad/s).
    tau : float or np.ndarray
        Generator torque (N m).

    Returns
    -------
    float or np.ndarray
        Generator loss power (W).
    """
    return 0.137 * np.abs(omega_g) + 3.118 * tau**2


def pe_loss_model(omega_g, tau):
    """
    Example power-electronics loss model.

    Parameters
    ----------
    omega_g : float or np.ndarray
        Generator-side angular speed (rad/s).
    tau : float or np.ndarray
        Generator torque (N m).

    Returns
    -------
    float or np.ndarray
        Power-electronics loss power (W).
    """
    return 5.204 + 0.2896 * np.abs(omega_g) + 0.3559 * np.abs(tau)


baseline_config = {
    # Turbine geometry and ratings
    "Radius": 0.5,                    # Rotor radius (m)
    "Prated": 1000.0,                 # Rated electrical power per turbine (W)
    "Trated": np.inf,                 # Rated generator torque (N m); no torque clipping in this example
    "dHub": 2.0,                      # Hub depth below free surface (m)
    "number_of_turbines": 2,          # Number of turbines in the system

    # Drivetrain and generator parameters
    "Ng": 30,                         # Gear ratio
    "Kt": 1.5,                        # Generator torque constant
    "Rw": 0.5,                        # Generator winding resistance
    "J_d": 1.0,                       # Drivetrain inertia
    "B_d": 0.01,                      # Drivetrain damping/friction
    "J_r": 1.0,                       # Rotor inertia

    # Power-conversion model
    "power_model": "generator_and_pe_loss_models",
    "generator_loss_model": generator_loss_model,
    "pe_loss_model": pe_loss_model,

    # Rotor performance functions
    "CpFunc": rotor.get_cp,
    "CqFunc": rotor.get_cq,
    "CtFunc": rotor.get_ct,
    "CpOpt": rotor.CpOpt,
    "TSROpt": rotor.TSROpt,
    "TSRmax": rotor.TSRmax,
}

print("Baseline turbine configuration defined.")
print(f"  Radius: {baseline_config['Radius']:.2f} m")
print(f"  Rated power per turbine: {baseline_config['Prated'] / 1000:.2f} kW")
print(f"  Hub depth: {baseline_config['dHub']:.2f} m")
print(f"  Number of turbines: {baseline_config['number_of_turbines']}")
print(f"  Power model: {baseline_config['power_model']}")

## 4. Define representative vessel/platform properties

The vessel/platform properties below are used for drag estimation and pitch-constraint checking.

These values are simplified representative inputs for this case study. They should be replaced with site-specific vessel, platform, or mooring data for design studies.

In [ ]:
user_vessel_properties = {
    "Xm": 5.77,                       # Horizontal force-application distance (m)
    "Zm": 1.65,                       # Vertical force-application distance (m)
    "Kphi": 1.95e6,                   # Pitch hydrostatic stiffness (N m/rad)
    "theta": np.deg2rad(45.0),        # Mooring line angle (rad)
    "phi": np.deg2rad(20.0),          # Representative pitch angle (rad)
    "area": 9.5,                      # Cross-sectional/projected area (m^2)
    "Cd": 1.0,                        # Drag coefficient
}

print("Representative vessel/platform properties:")
print(f"  Xm: {user_vessel_properties['Xm']:.2f} m")
print(f"  Zm: {user_vessel_properties['Zm']:.2f} m")
print(f"  Kphi: {user_vessel_properties['Kphi']:.3e} N m/rad")
print(f"  Mooring angle theta: {np.rad2deg(user_vessel_properties['theta']):.1f} deg")
print(f"  Pitch angle phi: {np.rad2deg(user_vessel_properties['phi']):.1f} deg")
print(f"  Area: {user_vessel_properties['area']:.2f} m^2")
print(f"  Cd: {user_vessel_properties['Cd']:.2f}")

## 5. Load and compare three Southeast Alaska tidal-current sites

This section loads three NOAA tidal-current sites and compares their basic resource metadata and flow-speed statistics.

The same baseline turbine configuration will later be evaluated at each site so we can see how site conditions affect performance, feasibility, and cost.

This case study retrieves tidal data directly from NOAA. 

In [ ]:
range_hrs = 14 * 24

# Use hourly data for documentation/runtime practicality.
# For higher-fidelity studies, reduce this value after confirming runtime.
time_step_s = 3600

city_data_file = "../data/AlaskaCityLatLong.txt"

site_specs = [
    {
        "label": "SEA0838",
        "station": "SEA0838",
        "startdate": "2020-01-01",
    },
    {
        "label": "SEA0819",
        "station": "SEA0819",
        "startdate": "2020-01-01",
    },
    {
        "label": "SEA0307",
        "station": "SEA0307",
        "startdate": "2020-01-01",
    },
]

sites = {}

for spec in site_specs:
    print(f"\nLoading site: {spec['label']}")

    try:
        tidal_site = process_tidal_data(
            station=spec["station"],
            startdate=spec["startdate"],
            range_hrs=range_hrs,
            time_step_s=time_step_s,
            city_data_file=city_data_file,
        )

        sites[spec["label"]] = tidal_site

        print(f"  Data source: {tidal_site.source}")
        print(f"  Station name: {tidal_site.station_name}")
        print(f"  Nearest city: {tidal_site.nearest_city}")
        print(f"  Mooring distance: {tidal_site.mooring_distance:.2f} m")
        print(f"  Cable length: {tidal_site.cable_length:.2f} m")
        print(f"  Latitude: {np.degrees(tidal_site.latitude):.4f} deg")
        print(f"  Longitude: {np.degrees(tidal_site.longitude):.4f} deg")
        print(f"  Number of time points: {len(tidal_site.times)}")
        print(f"  Mean flow speed: {np.mean(tidal_site.flow_speeds):.3f} m/s")
        print(f"  Max flow speed: {np.max(tidal_site.flow_speeds):.3f} m/s")

    except Exception as e:
        print(f"  Site {spec['label']} failed to load: {e}")

if not sites:
    raise RuntimeError("No tidal sites were loaded successfully.")

### Site summary

The table below summarizes the three Sitkana candidate sites. It includes basic site metadata and tidal-resource statistics used in the case study.

Because this is a non-grid-connected battery-charging example, the main focus is on tidal resource strength, mooring depth, and overall feasibility. Cable length is still reported as site metadata, but it is not the primary cost driver in the current battery-charging cost configuration.

The sites show different resource characteristics:

- ``SEA0838`` has the highest mean and maximum flow speed among the three sites.
- ``SEA0819`` has the weakest tidal resource but the deepest mooring depth.
- ``SEA0307`` has an intermediate tidal resource and mooring depth.

These differences help illustrate why the same turbine design can perform differently across candidate sites.

In [ ]:
site_summary = []

for label, tidal_site in sites.items():
    site_summary.append(
        {
            "Site": label,
            "Station": tidal_site.station_name,
            "Data source": tidal_site.source,
            "Nearest city": tidal_site.nearest_city,
            "Mooring distance (m)": tidal_site.mooring_distance,
            "Cable length (km)": tidal_site.cable_length / 1000,
            "Latitude (deg)": np.degrees(tidal_site.latitude),
            "Longitude (deg)": np.degrees(tidal_site.longitude),
            "Mean flow (m/s)": np.mean(tidal_site.flow_speeds),
            "Max flow (m/s)": np.max(tidal_site.flow_speeds),
            "Time points": len(tidal_site.times),
        }
    )

site_summary_df = pd.DataFrame(site_summary)

# Round for cleaner display
site_summary_df = site_summary_df.round(
    {
        "Mooring distance (m)": 2,
        "Cable length (km)": 2,
        "Latitude (deg)": 4,
        "Longitude (deg)": 4,
        "Mean flow (m/s)": 3,
        "Max flow (m/s)": 3,
    }
)

site_summary_df

## 6. Run baseline simulation for each site

The same baseline turbine configuration is evaluated at each site to enable a direct comparison of site-specific performance, constraints, and cost.

For each site, the workflow:

- updates the turbine configuration with site-specific tidal data and mooring depth,
- runs the rotor simulation,
- creates a representative vessel/platform object,
- checks physical constraints,
- estimates annual energy and LCOE for the battery-charging application.

The summary below shows how the same design performs differently across the candidate Sitkana sites.

In [ ]:
baseline_results = {}

for label, tidal_site in sites.items():
    print(f"\nRunning baseline simulation for {label} ({tidal_site.station_name})")

    config_site = baseline_config.copy()
    config_site["dMoor"] = tidal_site.mooring_distance
    config_site["Uinf"] = tidal_site.flow_speeds
    config_site["t"] = tidal_site.times

    rotor_sim = RotorSimulation(config_site)
    rotor_sim.simulate()
    result = rotor_sim.get_results()

    vessel = VesselData(
        user_defined=True,
        vessel_properties=user_vessel_properties,
        simResult=result,
    )

    constraint_checker = ConstraintChecker(rotor, config_site, vessel, result)

    constraints = {
        "Power Constraint": constraint_checker.check_power_constraint(),
        "Depth Constraint": constraint_checker.check_depth_constraint(),
        "Cavitation Constraint": constraint_checker.check_cavitation_constraint(),
        "Pitch Constraint": constraint_checker.check_pitch_constraint(),
    }

    constraint_margins = {
        "Min power margin (W)": np.min(constraint_checker.power_constraint()),
        "Depth margin (m)": constraint_checker.depth_constraint(),
        "Min cavitation margin (Pa)": np.min(constraint_checker.cavitation_constraint()),
        "Min pitch margin (N m)": np.min(constraint_checker.pitch_constraint()),
    }

    feasible = all(constraints.values())

    lcoe_data = LCOEData(
        tidalData=tidal_site,
        turbineConfig=config_site,
        vesselData=vessel,
        simResult=result,
        lifetime=10,
        discount_rate=0.1,
        turbulence_intensity=0.0,
        customer="customer_B",
        application="battery_charging",
        BatteryCapacity_kWh=10.0,
    )

    calc_base = LCOECalculator(lcoe_data)

    capex_summary = calc_base.get_capex_summary()
    adjusted_capex = capex_summary["adjusted_capex"]
    total_opex = calc_base.calculate_total_opex(adjusted_capex)
    annual_energy = calc_base.calculate_annual_energy()
    lcoe = calc_base.calculate_lcoe()

    baseline_results[label] = {
        "tidal": tidal_site,
        "config": config_site,
        "result": result,
        "vessel": vessel,
        "constraints": constraints,
        "constraint_margins": constraint_margins,
        "feasible": feasible,
        "calculator": calc_base,
        "lcoe": lcoe,
        "annual_energy": annual_energy,
        "adjusted_capex": adjusted_capex,
        "total_opex": total_opex,
        "mean_pac": np.mean(result["Pac"]),
        "mean_pbat": np.mean(result["Pbat"]),
        "max_pbat": np.max(result["Pbat"]),
    }

    print(
        f"  LCOE: ${lcoe:.4f}/kWh | "
        f"Annual energy: {annual_energy:.2f} kWh/year | "
        f"Mean Pbat: {np.mean(result['Pbat']) / 1000:.3f} kW | "
        f"Feasible: {feasible}"
    )

In [ ]:
baseline_summary = []

hours_per_year = 8760

for label, out in baseline_results.items():
    result = out["result"]

    battery_annual_energy = (np.mean(result["Pbat"]) / 1000) * hours_per_year

    baseline_summary.append(
        {
            "Site": label,
            "Station": out["tidal"].station_name,
            "LCOE ($/kWh)": out["lcoe"],

            # Energy used by the current LCOE workflow
            "LCOE energy, Pelec/Pac (kWh/yr)": out["annual_energy"],

            # Battery-side delivered energy estimate
            "Battery energy, Pbat (kWh/yr)": battery_annual_energy,

            # Cost summary
            "Adjusted CAPEX ($)": out["adjusted_capex"],
            "Total OPEX ($/yr)": out["total_opex"],

            # Power summary
            "Mean Pac (kW)": np.mean(result["Pac"]) / 1000,
            "Mean Pbat (kW)": np.mean(result["Pbat"]) / 1000,
            "Max Pbat (kW)": np.max(result["Pbat"]) / 1000,

            # Feasibility summary
            "Feasible": all(out["constraints"].values()),
            "Power constraint": out["constraints"]["Power Constraint"],
            "Depth constraint": out["constraints"]["Depth Constraint"],
            "Cavitation constraint": out["constraints"]["Cavitation Constraint"],
            "Pitch constraint": out["constraints"]["Pitch Constraint"],
        }
    )

baseline_summary_df = pd.DataFrame(baseline_summary)

baseline_summary_df = baseline_summary_df.round(
    {
        "LCOE ($/kWh)": 4,
        "LCOE energy, Pelec/Pac (kWh/yr)": 2,
        "Battery energy, Pbat (kWh/yr)": 2,
        "Adjusted CAPEX ($)": 2,
        "Total OPEX ($/yr)": 2,
        "Mean Pac (kW)": 4,
        "Mean Pbat (kW)": 4,
        "Max Pbat (kW)": 4,
    }
)

baseline_summary_df

### Baseline site-comparison interpretation

The same baseline design is feasible at all three Sitkana candidate sites, but the predicted performance differs substantially.

``SEA0838`` has the strongest tidal resource and produces the most annual energy. It also has the lowest baseline LCOE among the three sites.

``SEA0819`` has the weakest tidal resource. As a result, annual energy production is much lower and the baseline LCOE is much higher.

``SEA0307`` falls between the other two sites in both energy production and LCOE.

Because this is a battery-charging case study, the table reports two energy metrics:

- ``LCOE energy, Pelec/Pac``: the annual energy used by the current LCOE calculation.
- ``Battery energy, Pbat``: an estimate of delivered battery-side annual energy after power-electronics losses.

The difference between these values highlights the effect of power-conversion losses. For battery-delivered-energy studies, users should carefully consider whether ``Pbat`` should be used as the relevant energy metric.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))

baseline_summary_df.plot(
    x="Site",
    y="LCOE energy, Pelec/Pac (kWh/yr)",
    kind="bar",
    ax=ax[0],
    legend=False,
    color="tab:blue",
)
ax[0].set_ylabel("Energy (kWh/year)")
ax[0].set_title("LCOE energy basis\n(Pelec/Pac)")
ax[0].grid(True, axis="y", alpha=0.3)

baseline_summary_df.plot(
    x="Site",
    y="Battery energy, Pbat (kWh/yr)",
    kind="bar",
    ax=ax[1],
    legend=False,
    color="tab:green",
)
ax[1].set_ylabel("Energy (kWh/year)")
ax[1].set_title("Battery-side energy\n(Pbat)")
ax[1].grid(True, axis="y", alpha=0.3)

baseline_summary_df.plot(
    x="Site",
    y="LCOE ($/kWh)",
    kind="bar",
    ax=ax[2],
    legend=False,
    color="tab:orange",
)
ax[2].set_ylabel("LCOE ($/kWh)")
ax[2].set_title("Baseline LCOE")
ax[2].grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

### Interpretation

The same baseline design performs very differently across the three Sitkana sites. ``SEA0838`` produces the most energy and the lowest LCOE, while ``SEA0819`` produces much less energy and therefore a much higher LCOE. ``SEA0307`` falls between the two.

This illustrates an important point: site selection matters as much as turbine design. Even when the turbine configuration is held fixed, tidal resource strength strongly affects energy production and cost.

Because this is a battery-charging case study, the table reports both the energy used by the current LCOE calculation and an estimated battery-side delivered energy:

- ``LCOE energy, Pelec/Pac``: annual energy used by the current LCOE workflow.
- ``Battery energy, Pbat``: estimated annual energy delivered after power-electronics losses.

The difference between these values highlights the effect of power-conversion losses. Users interested specifically in delivered battery energy should review whether ``Pbat`` is the more appropriate energy metric for their analysis.

The values shown here are representative outputs used to demonstrate the software workflow and are not final engineering estimates.

## 7. Optimization across all three sites

Next, we optimize the baseline design for each of the three Sitkana candidate sites.

The optimizer performs an explicit grid search over selected design variables. In this case study, we vary:

- rotor radius,
- rated power.

For simplicity, hub depth and turbine count are held fixed.

For each candidate design, the optimizer:

1. updates the turbine configuration,
2. runs the rotor simulation,
3. checks physical constraints,
4. calculates LCOE for feasible designs,
5. identifies the lowest-LCOE feasible design within the selected search bounds.

This helps show how the preferred design changes with site conditions and how the battery-charging cost model responds to different turbine sizes.

In [ ]:
variable_bounds = {
    "Radius": (0.5, 2.0, 0.5),          # 0.5, 1.0, 1.5, 2.0 m
    "Prated": (500.0, 1500.0, 500.0),   # 500, 1000, 1500 W
}

fixed_params = {
    "dHub": baseline_config["dHub"],
    "number_of_turbines": baseline_config["number_of_turbines"],
}

# Count grid points for user awareness
num_grid_points = 1
for low, high, step in variable_bounds.values():
    values = np.arange(low, high + 0.5 * step, step)
    num_grid_points *= len(values)

print(f"Optimization grid size per site: {num_grid_points} design points")

opt_results = {}

for label, tidal_site in sites.items():
    print(f"\n=== Running optimization for {label} ({tidal_site.station_name}) ===")

    optimizer = LCOEOptimizer(
        tidal=tidal_site,
        rotor=rotor,
        base_config=baseline_config,
        user_vessel_properties=user_vessel_properties,
    )

    site_params = {
        "dMoor": tidal_site.mooring_distance,
        "Uinf": tidal_site.flow_speeds,
        "t": tidal_site.times,
    }

    try:
        # Suppress detailed per-design-point output for a cleaner case-study notebook.
        with contextlib.redirect_stdout(io.StringIO()):
            opt_result = optimizer.optimize(
                variable_bounds=variable_bounds,
                fixed_params=fixed_params,
                site_params=site_params,
                customer="customer_B",
                application="battery_charging",
                BatteryCapacity_kWh=10.0,
                lifetime=10,
                discount_rate=0.1,
                turbulence_intensity=0.0,
            )

        opt_results[label] = {
            "optimizer": optimizer,
            "result": opt_result,
        }

        print(f"  Optimal LCOE: ${opt_result['optimal_lcoe']:.4f}/kWh")
        print(
            f"  Feasible designs: "
            f"{len(opt_result['feasible_table'])} of {len(opt_result['results_table'])}"
        )

        print("  Optimal variable values:")
        for name, value in opt_result["optimal_params"].items():
            print(f"    {name}: {value}")

    except Exception as e:
        print(f"  Optimization failed for {label}: {e}")
        opt_results[label] = {
            "optimizer": optimizer,
            "result": None,
        }

## 8. Re-evaluate optimized designs

The optimizer returns the best design variables found within the selected search bounds. Here we re-evaluate each optimized design so we can summarize:

- annual energy used by the current LCOE workflow,
- battery-side annual energy based on ``Pbat``,
- CAPEX and OPEX,
- constraint status,
- and key optimized design parameters.

In this battery-charging case study, the current LCOE calculation uses ``Pelec`` for annual energy. In ``generator_and_pe_loss_models``, ``Pelec`` is treated as AC-side power, ``Pac``. Battery-side delivered power is reported separately as ``Pbat``.

In [ ]:
optimized_performance = {}

hours_per_year = 8760

for label, tidal_site in sites.items():
    opt = opt_results[label]["result"]

    if opt is None:
        print(f"\nSkipping {label}: no optimization result available.")
        continue

    opt_params = opt["optimal_params"]

    config_site = baseline_config.copy()
    config_site.update(
        {
            "Radius": opt_params["Radius"],
            "Prated": opt_params["Prated"],
            "dHub": baseline_config["dHub"],
            "number_of_turbines": baseline_config["number_of_turbines"],
            "dMoor": tidal_site.mooring_distance,
            "Uinf": tidal_site.flow_speeds,
            "t": tidal_site.times,
        }
    )

    rotor_sim = RotorSimulation(config_site)
    rotor_sim.simulate()
    result = rotor_sim.get_results()

    vessel = VesselData(
        user_defined=True,
        vessel_properties=user_vessel_properties,
        simResult=result,
    )

    constraint_checker = ConstraintChecker(rotor, config_site, vessel, result)

    constraints = {
        "Power Constraint": constraint_checker.check_power_constraint(),
        "Depth Constraint": constraint_checker.check_depth_constraint(),
        "Cavitation Constraint": constraint_checker.check_cavitation_constraint(),
        "Pitch Constraint": constraint_checker.check_pitch_constraint(),
    }

    constraint_margins = {
        "Min power margin (W)": np.min(constraint_checker.power_constraint()),
        "Depth margin (m)": constraint_checker.depth_constraint(),
        "Min cavitation margin (Pa)": np.min(constraint_checker.cavitation_constraint()),
        "Min pitch margin (N m)": np.min(constraint_checker.pitch_constraint()),
    }

    feasible = all(constraints.values())

    lcoe_data = LCOEData(
        tidalData=tidal_site,
        turbineConfig=config_site,
        vesselData=vessel,
        simResult=result,
        lifetime=10,
        discount_rate=0.1,
        turbulence_intensity=0.0,
        customer="customer_B",
        application="battery_charging",
        BatteryCapacity_kWh=10.0,
    )

    calculator = LCOECalculator(lcoe_data)

    adjusted_capex = calculator.calculate_total_capex()
    total_opex = calculator.calculate_total_opex(adjusted_capex)
    annual_energy = calculator.calculate_annual_energy()
    lcoe = calculator.calculate_lcoe()

    battery_annual_energy = (np.mean(result["Pbat"]) / 1000) * hours_per_year

    optimized_performance[label] = {
        "tidal": tidal_site,
        "config": config_site,
        "result": result,
        "vessel": vessel,
        "constraints": constraints,
        "constraint_margins": constraint_margins,
        "feasible": feasible,
        "calculator": calculator,
        "lcoe": lcoe,
        "annual_energy": annual_energy,
        "battery_annual_energy": battery_annual_energy,
        "adjusted_capex": adjusted_capex,
        "total_opex": total_opex,
        "mean_pac": np.mean(result["Pac"]),
        "mean_pbat": np.mean(result["Pbat"]),
        "max_pbat": np.max(result["Pbat"]),
    }

    print(f"\nOptimized results for {label}")
    print(f"  Station: {tidal_site.station_name}")
    print(f"  Radius: {config_site['Radius']:.2f} m")
    print(f"  Rated power per turbine: {config_site['Prated']:.1f} W")
    print(f"  Number of turbines: {config_site['number_of_turbines']}")
    print(f"  LCOE: ${lcoe:.4f}/kWh")
    print(f"  LCOE energy, based on Pelec/Pac: {annual_energy:.2f} kWh/year")
    print(f"  Battery-side energy, based on Pbat: {battery_annual_energy:.2f} kWh/year")
    print(f"  Mean Pbat: {np.mean(result['Pbat']) / 1000:.4f} kW")
    print(f"  Adjusted CAPEX: ${adjusted_capex:,.2f}")
    print(f"  Total OPEX: ${total_opex:,.2f} per year")
    print(f"  Feasible: {feasible}")

    for name, satisfied in constraints.items():
        print(f"  {name}: {'Satisfied' if satisfied else 'Not Satisfied'}")

### Baseline vs optimized comparison across all three sites

The optimization varies rotor radius and rated power while holding hub depth and turbine count fixed.

The table below compares the baseline and optimized designs at each site, including:

- design variables,
- LCOE,
- annual energy used by the current LCOE workflow,
- estimated battery-side annual energy,
- and constraint status.

For this battery-charging case study, the current LCOE calculation uses ``Pelec``, which is treated as AC-side power ``Pac`` in the selected loss-model mode. Battery-side delivered energy is reported separately using ``Pbat``.

In [ ]:
comparison_rows = []

for label in sites.keys():
    baseline = baseline_results[label]
    optimized = optimized_performance[label]
    opt_params = opt_results[label]["result"]["optimal_params"]

    comparison_rows.append(
        {
            "Site": label,
            "Station": baseline["tidal"].station_name,

            # Design variables
            "Baseline Radius (m)": baseline["config"]["Radius"],
            "Optimized Radius (m)": opt_params["Radius"],

            "Baseline Prated (W)": baseline["config"]["Prated"],
            "Optimized Prated (W)": opt_params["Prated"],

            "Baseline dHub (m)": baseline["config"]["dHub"],
            "Optimized dHub (m)": optimized["config"]["dHub"],

            "Baseline number of turbines": baseline["config"]["number_of_turbines"],
            "Optimized number of turbines": optimized["config"]["number_of_turbines"],

            # LCOE
            "Baseline LCOE ($/kWh)": baseline["lcoe"],
            "Optimized LCOE ($/kWh)": optimized["lcoe"],

            # Energy basis used by current LCOE workflow
            "Baseline LCOE energy, Pelec/Pac (kWh/yr)": baseline["annual_energy"],
            "Optimized LCOE energy, Pelec/Pac (kWh/yr)": optimized["annual_energy"],

            # Battery-side delivered energy
            "Baseline battery energy, Pbat (kWh/yr)": baseline.get(
                "battery_annual_energy",
                (np.mean(baseline["result"]["Pbat"]) / 1000) * 8760,
            ),
            "Optimized battery energy, Pbat (kWh/yr)": optimized["battery_annual_energy"],

            # Cost
            "Baseline adjusted CAPEX ($)": baseline["adjusted_capex"],
            "Optimized adjusted CAPEX ($)": optimized["adjusted_capex"],

            "Baseline OPEX ($/yr)": baseline["total_opex"],
            "Optimized OPEX ($/yr)": optimized["total_opex"],

            # Constraints
            "Optimized feasible": optimized["feasible"],
            "Power constraint": optimized["constraints"]["Power Constraint"],
            "Depth constraint": optimized["constraints"]["Depth Constraint"],
            "Cavitation constraint": optimized["constraints"]["Cavitation Constraint"],
            "Pitch constraint": optimized["constraints"]["Pitch Constraint"],
        }
    )

comparison_df = pd.DataFrame(comparison_rows)

comparison_df = comparison_df.round(
    {
        "Baseline Radius (m)": 2,
        "Optimized Radius (m)": 2,
        "Baseline Prated (W)": 1,
        "Optimized Prated (W)": 1,
        "Baseline dHub (m)": 2,
        "Optimized dHub (m)": 2,
        "Baseline LCOE ($/kWh)": 4,
        "Optimized LCOE ($/kWh)": 4,
        "Baseline LCOE energy, Pelec/Pac (kWh/yr)": 2,
        "Optimized LCOE energy, Pelec/Pac (kWh/yr)": 2,
        "Baseline battery energy, Pbat (kWh/yr)": 2,
        "Optimized battery energy, Pbat (kWh/yr)": 2,
        "Baseline adjusted CAPEX ($)": 2,
        "Optimized adjusted CAPEX ($)": 2,
        "Baseline OPEX ($/yr)": 2,
        "Optimized OPEX ($/yr)": 2,
    }
)

comparison_df

## 9. Compare baseline and optimized performance

The plots below compare the baseline and optimized designs across the three Sitkana sites.

Because this is a battery-charging case study, two energy metrics are shown:

- ``Pelec/Pac`` energy: the annual energy used by the current LCOE workflow.
- ``Pbat`` energy: estimated battery-side delivered energy after power-electronics losses.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 5))

x = np.arange(len(comparison_df["Site"]))
width = 0.35

# LCOE comparison
ax[0].bar(
    x - width / 2,
    comparison_df["Baseline LCOE ($/kWh)"],
    width,
    label="Baseline",
    color="tab:blue",
)
ax[0].bar(
    x + width / 2,
    comparison_df["Optimized LCOE ($/kWh)"],
    width,
    label="Optimized",
    color="tab:orange",
)
ax[0].set_xticks(x)
ax[0].set_xticklabels(comparison_df["Site"])
ax[0].set_ylabel("LCOE ($/kWh)")
ax[0].set_title("Baseline vs Optimized LCOE")
ax[0].legend()
ax[0].grid(True, axis="y", alpha=0.3)

# LCOE energy basis comparison
ax[1].bar(
    x - width / 2,
    comparison_df["Baseline LCOE energy, Pelec/Pac (kWh/yr)"],
    width,
    label="Baseline",
    color="tab:blue",
)
ax[1].bar(
    x + width / 2,
    comparison_df["Optimized LCOE energy, Pelec/Pac (kWh/yr)"],
    width,
    label="Optimized",
    color="tab:orange",
)
ax[1].set_xticks(x)
ax[1].set_xticklabels(comparison_df["Site"])
ax[1].set_ylabel("Energy (kWh/year)")
ax[1].set_title("LCOE Energy Basis\n(Pelec/Pac)")
ax[1].legend()
ax[1].grid(True, axis="y", alpha=0.3)

# Battery-side delivered energy comparison
ax[2].bar(
    x - width / 2,
    comparison_df["Baseline battery energy, Pbat (kWh/yr)"],
    width,
    label="Baseline",
    color="tab:blue",
)
ax[2].bar(
    x + width / 2,
    comparison_df["Optimized battery energy, Pbat (kWh/yr)"],
    width,
    label="Optimized",
    color="tab:orange",
)
ax[2].set_xticks(x)
ax[2].set_xticklabels(comparison_df["Site"])
ax[2].set_ylabel("Battery energy (kWh/year)")
ax[2].set_title("Battery-Side Energy\n(Pbat)")
ax[2].legend()
ax[2].grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

### Interpretation

For ``SEA0838``, the optimizer recovered the baseline design within the selected search bounds. This indicates that, for this site and cost model, the baseline rated power was already preferred among the tested options.

For ``SEA0819`` and ``SEA0307``, the optimizer reduced rated power from 1000 W to 500 W while keeping rotor radius, hub depth, and turbine count fixed. This reduced CAPEX and OPEX, which lowered LCOE under the current cost model.

The weaker ``SEA0819`` resource still produces very little battery-side energy, even after optimization. This highlights that reducing system cost can improve LCOE, but it does not necessarily make a weak tidal site attractive for battery charging.

The difference between ``Pelec/Pac`` energy and ``Pbat`` energy shows the impact of generator and power-electronics losses. For battery-charging applications, delivered battery-side energy should be reviewed carefully.

## 10. Summary

This case study compared three Southeast Alaska tidal-current sites using a common baseline battery-charging turbine design and a site-specific grid-search optimization.

The results highlight that:

- site tidal resource strongly affects annual energy production,
- weaker sites may favor lower rated-power designs under the selected cost model,
- physical constraints must be checked alongside cost,
- generator and power-electronics losses can substantially reduce delivered battery-side energy,
- optimization results depend on the selected search bounds, cost model, battery capacity, and loss-model assumptions.

The values shown here are screening-level estimates intended to demonstrate the VITAL workflow. They should not be interpreted as final engineering or deployment recommendations.